# 11 - Export of every table and every quoted number (workflow step 12)

No result number is typed by hand: Table 1 is generated from `dfm_acdc/config.py`, Tables 3-5
and the supplementary sensitivity table from the CSV files of notebooks 07-10, and every number
quoted in the Abstract / Results / Conclusion is collected in `output/results/key_numbers.csv`.

Outputs: `output/results/tables/Table1_parameters.tex`, `Table3_nominal.tex`, `Table4_monte_carlo.tex`,
`Table5_ablation.tex`, `TableS1_sensitivity.tex`, `output/results/key_numbers.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd

from dfm_acdc.config import (RESULTS_DIR, NetworkParams, RenewableParams, BESSParams, DegradationParams,
                             DFMParams, UncertaintyParams, EMSParams, MCParams)
from dfm_acdc.degradation import XuDegradation
from dfm_acdc.envelope import degradation_allowance

TAB = os.path.join(RESULTS_DIR, "tables"); os.makedirs(TAB, exist_ok=True)
n, r, b, dp, d, u, e, mc = (NetworkParams(), RenewableParams(), BESSParams(), DegradationParams(),
                            DFMParams(), UncertaintyParams(), EMSParams(), MCParams())
def write(name, body):
    open(os.path.join(TAB, name), "w", encoding="utf8").write(body)
    print(f"----- {name}\n{body}")

## Table 1 -- parameters (from the configuration)

In [ ]:
deg = XuDegradation(dp, b.soc_max, b.soc_min)
lam_deg = e.bess_cost_eur_per_kwh * b.E_rated_kwh / (1 - dp.soh_eol)
rows = [
    ("Network", "Nominal AC voltage", r"400~V (231~V phase-to-neutral)"),
    ("Network", "Nominal DC voltage", f"{n.V_dc_nom:.0f}~V"),
    ("Network", "Transformers", r"20/0.4~kV; 500/150/300~kVA (R/I/C); $R_{cc}=1\%$, $X_{cc}=4\%$"),
    ("Network", "DC-line resistance", r"0.162~$\Omega$/km (UG)"),
    ("Network", "DC-line current rating", f"{n.I_dc_line_max:.0f}~A"),
    ("Network", "AC voltage limits", rf"{n.V_min_pu:.2f}--{n.V_max_pu:.2f}~pu (phase-to-neutral)"),
    ("Network", "DC voltage limits", rf"{n.V_dc_min_pu:.2f}--{n.V_dc_max_pu:.2f}~pu"),
    ("Network", "Feeder-head current limits", r"724.6 / 231.9 / 434.8~A (R01/I01/C01)"),
    ("Network", "VSCs", r"8 P--V droop ($K_{droop}=10^6$) + 1 $V_{dc}$--$Q$ (I01--H01)"),
    ("Network", "VSC ratings", r"R01, C01, I01: 150; R10, I02, C11: 50; R14, C09, C16: 20~kVA"),
    ("Renewables", "DC-side PV", f"{r.pv_rated_kwp:.0f}~kWp on 10 DC load buses"),
    ("Renewables", "DC-side wind turbine", f"{r.wt_rated_kw:.0f}~kW at {r.wt_bus}"),
    ("BESS", "Location", f"DC hub bus {n.bess_bus}"),
    ("BESS", "Rated power", f"{b.P_rated_kw:.0f}~kW"),
    ("BESS", "Rated energy", f"{b.E_rated_kwh:.0f}~kWh"),
    ("BESS", r"$SOC_{\min}/SOC_{\max}$", f"{b.soc_min:.1f} / {b.soc_max:.1f}"),
    ("BESS", "Initial SOC", f"{b.soc_0:.1f}"),
    ("BESS", "Initial SOH", f"{b.soh_0:.2f}"),
    ("BESS", r"$\eta_{\mathrm{ch}}/\eta_{\mathrm{dis}}$", f"{b.eta_ch:.2f} / {b.eta_dis:.2f}"),
    ("BESS", "Aging model", r"Xu et al. (LMO), $SOH_{EOL}=0.8$"),
    ("DFM", r"Support horizon $H_f$", f"{d.H_f_h*60:.0f}~min"),
    ("DFM", r"Risk tolerance $\varepsilon$", f"{d.eps_risk:.2f}"),
    ("DFM", "Degradation allowance", rf"{d.N_ref_cycles_per_day:g} reference cycle/day ({d.DoD_ref*100:.0f}\% DoD)"),
    ("Uncertainty", "Forecast-error std.", rf"PV {u.sigma_pv_rel*100:.0f}\%, wind {u.sigma_wt_rel*100:.0f}\%+{u.sigma_wt_abs_pu*100:.0f}\%$P_r$, demand {u.sigma_load_rel*100:.0f}\%; AR(1) $\phi={u.ar1_phi}$"),
    ("EMS", "Sampling interval", f"{e.dt_h*60:.0f}~min"),
    ("EMS", "Prediction horizon", r"to end of day ($\leq$24~h); 15-min stages for 4~h, then 1-h blocks"),
    ("EMS", "Tariff", rf"{e.price_valley:.2f}/{e.price_flat:.2f}/{e.price_peak:.2f}~EUR/kWh (valley/flat/peak)"),
    ("EMS", r"$\lambda_{\mathrm{deg}}$, $\lambda_{\mathrm{curt}}$", rf"{lam_deg/1e3:.0f}~kEUR per unit $D$; {e.lambda_curt:.2f}~EUR/kWh"),
    ("EMS", "Solver", "MILP, HiGHS (SciPy 1.15)"),
    ("EMS", "Solver tolerance", rf"relative MIP gap $10^{{{int(np.log10(e.mip_rel_gap))}}}$"),
    ("Evaluation", "Monte Carlo trials", f"{mc.n_trials}"),
]
body = "\\begin{tabular}{lll}\n\\toprule\nCategory & Parameter & Value \\\\\n\\midrule\n"
last = None
for cat, par, val in rows:
    if last is not None and cat != last:
        body += "\\midrule\n"
    body += f"{cat} & {par} & {val} \\\\\n"
    last = cat
body += "\\bottomrule\n\\end{tabular}\n"
write("Table1_parameters.tex", body)

## Table 3 -- deterministic 24-h performance

In [ ]:
nom = pd.read_csv(os.path.join(RESULTS_DIR, "results_nominal_summary.csv"), index_col=0)
def fmt_soh(x):
    return f"{x*1e5:.2f}"
body = ("\\begin{tabular}{lcccccccc}\n\\toprule\n"
        "Strategy & $E_{\\mathrm{grid}}$ & Cost & $E_{\\mathrm{curt}}$ & $\\eta_{\\mathrm{RES}}$ & $E_{\\mathrm{loss}}$ & "
        "$\\Delta SOH$ & $N_{\\mathrm{EFC}}$ & Viol. \\\\\n"
        " & [kWh] & [EUR] & [kWh] & [\\%] & [kWh] & [$10^{-5}$] & [--] & [int.] \\\\\n\\midrule\n")
for s in ("B0", "B1", "B2", "DFM"):
    x = nom.loc[s]
    soh = "--" if s == "B0" else fmt_soh(x.dSOH_rainflow)
    efc = "--" if s == "B0" else f"{x.N_EFC:.2f}"
    body += (f"{s} & {x.E_grid_corr_kWh:.0f} & {x.cost_corr_EUR:.1f} & {x.E_curt_kWh:.1f} & {x.eta_RES*100:.1f} & "
             f"{x.E_loss_kWh:.1f} & {soh} & {efc} & {int(x.n_viol_intervals)} \\\\\n")
body += "\\bottomrule\n\\end{tabular}\n"
write("Table3_nominal.tex", body)

## Table 4 -- Monte Carlo statistics

In [ ]:
ms = pd.read_csv(os.path.join(RESULTS_DIR, "results_mc_summary.csv"))
def g(s, m, stat):
    return ms[(ms.strategy == s) & (ms.metric == m)][stat].iloc[0]
spec = [("$E_{\\mathrm{grid}}$ [kWh]", "E_grid_corr_kWh", ("mean", "P95"), "{:.0f}", 1),
        ("Cost [EUR]", "cost_corr_EUR", ("mean", "P95"), "{:.1f}", 1),
        ("$E_{\\mathrm{curt}}$ [kWh]", "E_curt_kWh", ("mean", "P95"), "{:.1f}", 1),
        ("$\\Delta SOH$ [$10^{-5}$]", "dSOH_rainflow", ("mean", "P95"), "{:.2f}", 1e5),
        ("Uncovered imbalance [kWh]", "E_dev_kWh", ("mean", "P95"), "{:.1f}", 1),
        ("Max. exceedance [\\%]", "max_exceedance", ("P95",), "{:.2f}", 100),
        ("Trials with violation [\\%]", "P_viol_trials", ("mean",), "{:.0f}", 100)]
body = "\\begin{tabular}{llcccc}\n\\toprule\nMetric & Statistic & B0 & B1 & B2 & DFM \\\\\n\\midrule\n"
for lab, m, stats, f, sc in spec:
    for i, st in enumerate(stats):
        vals = []
        for s in ("B0", "B1", "B2", "DFM"):
            if m == "dSOH_rainflow" and s == "B0":
                vals.append("--"); continue
            v = g(s, m, st) * sc
            vals.append(f.format(v))
        name = {"mean": "Mean", "P95": "P95"}[st] if m != "P_viol_trials" else "--"
        body += f"{lab if i == 0 else ''} & {name} & " + " & ".join(vals) + " \\\\\n"
body += "\\bottomrule\n\\end{tabular}\n"
write("Table4_monte_carlo.tex", body)

## Table 5 -- ablation and computational burden

In [ ]:
ab = pd.read_csv(os.path.join(RESULTS_DIR, "results_ablation_summary.csv"), index_col=0)
body = ("\\begin{tabular}{lccccccc}\n\\toprule\n"
        "Configuration & $E_{\\mathrm{grid}}$ & $E_{\\mathrm{curt}}$ & $\\Delta SOH$ & Imbalance & Viol. trials & "
        "Mean solve & P95 solve \\\\\n"
        " & [kWh] & [kWh] & [$10^{-5}$] & [kWh] & [\\%] & [ms] & [ms] \\\\\n\\midrule\n")
names = {"M0": "$\\mathcal{M}_0$ (full DFM)", "M1": "$\\mathcal{M}_1$ (no health/deg.)",
         "M2": "$\\mathcal{M}_2$ (no reserve)", "M3": "$\\mathcal{M}_3$ (no network)"}
for c in ("M0", "M1", "M2", "M3"):
    x = ab.loc[c]
    body += (f"{names[c]} & {x.mc_mean_E_grid_corr_kWh:.0f} & {x.mc_mean_E_curt_kWh:.1f} & {x.mc_mean_dSOH_rainflow*1e5:.2f} & "
             f"{x.mc_mean_E_dev_kWh:.1f} & {x.mc_P_viol_trials*100:.0f} & {x.mc_mean_t_solve_mean_ms:.0f} & "
             f"{x.mc_t_solve_p95_ms_pooled:.0f} \\\\\n")
body += "\\bottomrule\n\\end{tabular}\n"
write("Table5_ablation.tex", body)

## Supplementary Table S1 -- sensitivity (Monte Carlo means)

In [ ]:
se = pd.read_csv(os.path.join(RESULTS_DIR, "results_sensitivity_summary.csv"))
plab = {"H_f_h": "$H_f$ [min]", "eps_risk": "$\\varepsilon$", "soh_0": "$SOH_0$", "N_ref_cycles_per_day": "$N_{ref}$"}
body = ("\\begin{tabular}{llcccccc}\n\\toprule\n"
        "Parameter & Value & Cost & $E_{\\mathrm{curt}}$ & $\\Delta SOH$ & Imbalance & $\\bar P^{dis}_{safe}$ & Deg.-limited \\\\\n"
        " & & [EUR] & [kWh] & [$10^{-5}$] & [kWh] & [kW] & [\\% int.] \\\\\n\\midrule\n")
for par in ("H_f_h", "eps_risk", "soh_0", "N_ref_cycles_per_day"):
    dd = se[se.parameter == par].sort_values("value")
    for i, (_, x) in enumerate(dd.iterrows()):
        v = f"{x.value*60:.0f}" if par == "H_f_h" else f"{x.value:g}"
        body += (f"{plab[par] if i == 0 else ''} & {v} & {x.mc_mean_cost_corr_EUR:.1f} & {x.mc_mean_E_curt_kWh:.1f} & "
                 f"{x.mc_mean_dSOH_rainflow*1e5:.2f} & {x.mc_mean_E_dev_kWh:.1f} & {x.mc_mean_mean_P_safe_dis_kW:.1f} & "
                 f"{x.mc_mean_share_lim_dis_deg*100:.0f} \\\\\n")
    body += "\\midrule\n" if par != "N_ref_cycles_per_day" else ""
body += "\\bottomrule\n\\end{tabular}\n"
write("TableS1_sensitivity.tex", body)

## Every number quoted in the text

In [ ]:
pt = pd.read_csv(os.path.join(RESULTS_DIR, "results_mc_paired_tests.csv"))
ev = pd.read_csv(os.path.join(RESULTS_DIR, "results_ems_validation.csv"))
L = pd.read_csv(os.path.join(RESULTS_DIR, "log_nominal_DFM.csv"))
L0 = pd.read_csv(os.path.join(RESULTS_DIR, "log_trial0_DFM.csv"))
pre = pd.read_csv(os.path.join(RESULTS_DIR, "precompute_network_forecast.csv"))
b0r = pd.read_csv(os.path.join(RESULTS_DIR, "results_B0_renewables.csv"))
pct = lambda a, b_: 100 * (a - b_) / b_
K = {}
for s in ("B1", "B2"):
    K[f"nom_cost_DFM_vs_{s}_pct"] = pct(nom.loc["DFM", "cost_corr_EUR"], nom.loc[s, "cost_corr_EUR"])
    K[f"nom_Egrid_DFM_vs_{s}_pct"] = pct(nom.loc["DFM", "E_grid_corr_kWh"], nom.loc[s, "E_grid_corr_kWh"])
    K[f"nom_curt_DFM_vs_{s}_pct"] = pct(nom.loc["DFM", "E_curt_kWh"], nom.loc[s, "E_curt_kWh"])
    K[f"nom_dSOH_DFM_vs_{s}_pct"] = pct(nom.loc["DFM", "dSOH_rainflow"], nom.loc[s, "dSOH_rainflow"])
K["nom_cost_DFM_vs_B0_pct"] = pct(nom.loc["DFM", "cost_corr_EUR"], nom.loc["B0", "cost_corr_EUR"])
K["nom_Egrid_DFM_vs_B0_pct"] = pct(nom.loc["DFM", "E_grid_corr_kWh"], nom.loc["B0", "E_grid_corr_kWh"])
K["nom_curt_DFM_vs_B0_pct"] = pct(nom.loc["DFM", "E_curt_kWh"], nom.loc["B0", "E_curt_kWh"])
for s in ("B0", "B1", "B2", "DFM"):
    for m in ("E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "N_EFC", "n_viol_intervals"):
        K[f"nom_{m}_{s}"] = nom.loc[s, m]
for _, x in pt.iterrows():
    K[f"mc_{x.metric}_DFM_vs_{x.vs}_pct"] = x.rel_diff_pct
    K[f"mc_{x.metric}_DFM_vs_{x.vs}_share_better"] = x.share_DFM_better
    K[f"mc_{x.metric}_DFM_vs_{x.vs}_wilcoxon_p"] = x.wilcoxon_p
for s in ("B0", "B1", "B2", "DFM"):
    K[f"mc_P_viol_trials_{s}"] = g(s, "P_viol_trials", "mean")
    K[f"mc_E_dev_mean_{s}"] = g(s, "E_dev_kWh", "mean")
for c in ("M1", "M2", "M3"):
    for m in ("cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "E_dev_kWh"):
        K[f"abl_rel_{m}_{c}_pct"] = ab.loc[c, f"rel_mc_{m}_pct"]
K["solve_time_DFM_mean_ms"] = nom.loc["DFM", "t_solve_mean_ms"]
K["solve_time_DFM_p95_ms"] = nom.loc["DFM", "t_solve_p95_ms"]
K["solve_time_DFM_max_ms"] = nom.loc["DFM", "t_solve_max_ms"]
K["solve_time_DFM_mc_p95_pooled_ms"] = ab.loc["M0", "mc_t_solve_p95_ms_pooled"]
K["mean_P_flex_dis_nominal_kW"] = L.st_P_flex_dis.mean(); K["mean_P_flex_ch_nominal_kW"] = L.st_P_flex_ch.mean()
K["mean_P_safe_dis_nominal_kW"] = L.st_P_safe_dis.mean(); K["mean_P_safe_ch_nominal_kW"] = L.st_P_safe_ch.mean()
K["share_rated_window_used_dis_pct"] = 100 * L.st_P_safe_dis.mean() / b.P_rated_kw
for side in ("dis", "ch"):
    for lim in ("en", "deg", "net", "rated"):
        K[f"nominal_share_lim_{side}_{lim}_pct"] = 100 * (L[f"st_lim_{side}"] == lim).mean()
        K[f"trial0_share_lim_{side}_{lim}_pct"] = 100 * (L0[f"st_lim_{side}"] == lim).mean()
K["B0_curt_kWh"] = float(b0r.P_curt_kW.sum() * e.dt_h); K["B0_curt_peak_kW"] = float(b0r.P_curt_kW.max())
K["Pnet_dis_min_midday_kW"] = float(pre[(pre.t_h >= 9) & (pre.t_h < 13)].P_net_dis.max())
K["Pnet_ch_min_evening_kW"] = float(pre.P_net_ch.min())
K["ems_validation_tests_passed"] = f"{int(ev['pass'].sum())}/{len(ev)}"
# --- robustness of the terminal-SOC correction (DFM may end the day below SOC_0 when its daily
# allowance is exhausted): (i) add the cycle aging of restoring SOC_0 (a charge half-cycle);
# (ii) value the restoration energy at the flat instead of the valley price.
tr = pd.read_csv(os.path.join(RESULTS_DIR, "results_mc_trials.csv"))
dL0 = float(deg.dL(deg.F_from_soh(b.soh_0)))
def restore_dsoh(soc_f):
    if soc_f >= b.soc_0:
        return 0.0
    return dL0 * 0.5 * float(deg.G(b.soc_max - soc_f) - deg.G(b.soc_max - b.soc_0))
tr["dSOH_incl_restore"] = tr.dSOH_rainflow + tr.soc_final.map(restore_dsoh)
tr["cost_corr_flat_EUR"] = tr.cost_corr_EUR + np.where(tr.dE_terminal_kWh > 0, tr.dE_terminal_kWh / b.eta_ch, tr.dE_terminal_kWh * b.eta_dis) * (e.price_flat - e.price_valley)
for s in ("B1", "B2", "DFM"):
    K[f"mc_dSOH_incl_restore_mean_{s}"] = tr[tr.strategy == s].dSOH_incl_restore.mean()
    K[f"mc_soc_final_mean_{s}"] = tr[tr.strategy == s].soc_final.mean()
for s in ("B0", "B1", "B2", "DFM"):
    K[f"mc_cost_corr_flat_mean_{s}"] = tr[tr.strategy == s].cost_corr_flat_EUR.mean()
for s in ("B1", "B2"):
    K[f"mc_dSOH_incl_restore_DFM_vs_{s}_pct"] = pct(K["mc_dSOH_incl_restore_mean_DFM"], K[f"mc_dSOH_incl_restore_mean_{s}"])
    K[f"mc_cost_corr_flat_DFM_vs_{s}_pct"] = pct(K["mc_cost_corr_flat_mean_DFM"], K[f"mc_cost_corr_flat_mean_{s}"])
K["mc_cost_corr_flat_DFM_vs_B0_pct"] = pct(K["mc_cost_corr_flat_mean_DFM"], K["mc_cost_corr_flat_mean_B0"])
for s in ("B0", "B1", "B2", "DFM"):
    for m in ("E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "E_dev_kWh", "N_EFC", "eta_RES"):
        K[f"mc_mean_{m}_{s}"] = g(s, m, "mean")
        K[f"mc_P95_{m}_{s}"] = g(s, m, "P95")
key = pd.DataFrame({"key": list(K.keys()), "value": list(K.values())})
key.to_csv(os.path.join(RESULTS_DIR, "key_numbers.csv"), index=False)
pd.set_option("display.max_rows", 300)
print(key.to_string(index=False))